In [ ]:
# --- Libraries ---
# dplyr      : tidy data manipulation
# Cairo      : nice anti-aliased font rendering for PNG output (ggplot)
# patchwork  : compose multiple ggplots into a single figure
library(dplyr)
library(Cairo)
library(patchwork)
library(GenomicRanges)

# Force ggplot/Cairo to use a real sans-serif font on this system,
# otherwise font lookup falls back to a 'sans' alias that is not always
# available inside Jupyter R kernels and the figure renders with boxes.
CairoFonts(
  regular     = "sans:style=Regular",
  bold        = "sans:style=Bold",
  italic      = "sans:style=Italic",
  bolditalic  = "sans:style=Bold Italic,BoldItalic",
  symbol      = "Symbol", usePUA = TRUE
)

# GM12878 heterozygous SNP density map

**Goal:** find the most informative 2 Mb genomic windows for allele-specific
analyses in GM12878 (NA12878). "Informative" here means *densely populated
with phased heterozygous SNVs* — these are the only positions where reads can
actually be assigned to a haplotype, so they bound how much allele-specific
signal you can recover from any given window.

**Pipeline (cell-by-cell):**

1. Load the **Illumina Platinum Genomes 2017 NA12878 phased VCF** (hg38).
   - Already statistically phased (`GT` like `0|1` / `1|0`); `PS` tag carries
     the phase set ID.
2. **Keep only useful sites:**
   - drop `1|1` homozygous-alt (not informative for haplotype assignment),
   - drop `1|2` / `2|1` triallelic (rare, complicates phasing),
   - drop indels, keep only SNVs (`nchar(ref)==nchar(alt)==1`).
3. Convert SNVs → `GRanges`. Convert a pre-computed bed of **10 kb tiling
   bins** of GRCh38 (`GRCh38_ref/GRCh38_10000_windows.bed`) → `GRanges`.
4. Overlap SNVs against 10 kb bins → number of phased het SNVs per 10 kb bin.
   Save as `GM12878_10k_snp_counts_summary.txt` and a bedGraph view.
5. Build **500 kb tiling bins**, count how many of the 50 child 10-kb bins
   inside each 500 kb bin actually contain ≥1 SNV (i.e. the 10-kb-resolution
   *coverage fraction*, not raw SNV count). Range 0–50.
6. **Slide a 4×500 kb (=2 Mb) window with step 500 kb** over the genome,
   compute `mean_value = mean(coverage_fraction across the 4 child bins)`.
   Sort descending → top windows are the genomic regions with the broadest
   spread of phased heterozygous SNVs at 10 kb resolution.
7. Save as **`GM12878_2M_10k_snp_density_summary.txt`**.

## What `GM12878_2M_10k_snp_density_summary.txt` contains

A genome-wide ranking of overlapping 2 Mb windows by SNV-coverage density:

| column | type | meaning |
|---|---|---|
| `chrom` | chr | chromosome (chr1–22, chrX) |
| `start` | int | window start, hg38, 0-based |
| `end`   | int | `start + 2 000 000` |
| `mean_value` | float | average # of SNV-bearing 10-kb bins per 500 kb sub-bin within this 2 Mb window. Range 0–50; higher = denser SNV coverage = more haplotype-informative window |

Rows are sorted by `mean_value` descending, so `head -n N` gives the top-N
most SNV-dense 2 Mb windows. Used by `corigami` plan-A/plan-B benchmarks to
pick the regions where allele-specific Hi-C prediction is even meaningful.

In [ ]:
# Use repository-local phased variants and reference metadata.
repo_root <- normalizePath("..")
vcf_path <- file.path(repo_root, "src/data/variants/illumina_PlatinumGenomes_2017_hg38_NA12878_PS.vcf.gz")
chrom_sizes_path <- file.path(repo_root, "src/data/reference/GRCh38.chrom.sizes")
summary_path <- file.path(repo_root, "src/data/regions/GM12878_2M_10k_snp_density_summary.txt")
snp <- read.table(vcf_path)


In [ ]:
# Standard VCF column layout. The trailing 'sample' column packs the
# per-sample fields described by 'format' (e.g. "GT:PS" → "1|0:1000").
colnames(snp) <- c("chrom","pos","id","ref","alt","qual","filter","info","format","sample")

In [ ]:
# Pull just the genotype string (GT field) out of the packed sample column.
# 'sample' looks like "1|0:1000" where the part before ':' is GT and after ':'
# is the phase set (PS) tag. We only need GT here.
snp$GT <- gsub(":.*","",snp$sample)

In [ ]:
# Sanity check: how many sites of each genotype class?
#   0|1, 1|0  → phased heterozygous (informative for haplotype assignment)
#   1|1       → homozygous-alt (NOT informative — both haplotypes look the same)
#   1|2, 2|1  → phased multi-allelic heterozygous (rare)
table(snp$GT)

In [ ]:
# Drop homozygous-alt (1|1). Reads covering these positions cannot be
# attributed to a specific haplotype because both maternal and paternal
# carry the alt allele. Keep 0|1, 1|0, and the rare 1|2 / 2|1.
snp <- snp[snp$GT != "1|1",]

In [ ]:
# Compute REF / ALT allele lengths so we can keep only single-nucleotide
# substitutions in the next cell. Indels (length != 1) are excluded because:
#   1) downstream tools we use (bcftools consensus / WhatsHap haplotag) handle
#      SNVs and indels differently — restricting to SNVs keeps coordinates
#      identical between hg38 and the haplotype-resolved consensus,
#   2) indel callsets are noisier than SNV callsets in this VCF.
ref_len <- lapply(1:nrow(snp), function(x) nchar(snp$ref[x])) %>% unlist()
alt_len <- lapply(1:nrow(snp), function(x) nchar(snp$alt[x])) %>% unlist()

In [ ]:
# Keep biallelic SNVs (single base on both REF and ALT). After this filter
# we have ~2.17M phased het SNVs across autosomes + chrX.
snp_used <- snp[ref_len == 1 & alt_len == 1,]

In [ ]:
# Quick visual check of the kept rows.
head(snp_used)

In [ ]:
# Confirm chromosome coverage — autosomes 1–22 + chrX (no chrY in this VCF
# because NA12878 is female, no chrM either).
unique(snp_used$chrom)

In [ ]:
# Convert the SNV table to a GRanges using gUtils::dt2gr.
# dt2gr looks for chrom + (start|pos) + (end|pos) columns; here we only have
# 'pos', so it builds zero-width ranges at each SNV position. The metadata
# columns (ref, alt, GT, ...) ride along as mcols on the GRanges.
library(gUtils)
snp_used_gr <- dt2gr(snp_used)

In [ ]:
# Inspect: should report ~2.17M ranges, 23 sequences (autosomes + X).
snp_used_gr

In [ ]:
# Build 10 kb GRCh38 tiles directly from the uploaded chromosome sizes.
chrom_sizes <- read.table(chrom_sizes_path, col.names = c("chrom", "length"))
seqinfo_hg38 <- Seqinfo(
  seqnames = chrom_sizes$chrom,
  seqlengths = chrom_sizes$length
)
windows_10k <- tileGenome(
  seqinfo_hg38,
  tilewidth = 10000,
  cut.last.tile.in.chrom = TRUE
)
bins <- as.data.frame(windows_10k)[, c("seqnames", "start", "end")]
colnames(bins) <- c("chrom", "start", "end")
bins$start <- bins$start - 1L
bins$idx <- seq_len(nrow(bins))


In [ ]:
# Convert the generated 10 kb table to the GRanges layout used below.
bins_gr <- dt2gr(bins)


In [ ]:
# Overlap 10 kb bins (query) with SNV positions (subject).
# `gUtils::%*%` is the GRanges-overlap operator: it returns one row per
# (query, subject) pair that overlaps. The result `ov` carries:
#   - query.id    = bin index that the SNV falls in
#   - subject.id  = SNV index
#   - all original mcols from both sides (ref, alt, GT, idx, ...)
ov <- bins_gr %*% snp_used_gr

In [ ]:
# Inspect: ~2.17M overlap rows. Slightly more rows than total SNVs because
# a bin boundary can split a (zero-width) SNV onto two adjacent bins in
# 1-based vs 0-based coord overlap.
ov

In [ ]:
# Per-bin SNV count: tabulate overlap rows by bin (query.id).
# `Var1` (factor of bin index) is converted back to integer for joining later.
ov_sum <- table(ov$query.id) %>% as.data.frame()
ov_sum$Var1 <- as.integer(as.character(ov_sum$Var1))

In [ ]:
# Join the SNV counts back onto the full set of 10 kb bins so that bins
# WITHOUT any SNV are still represented as rows (with Freq=NA, fixed below).
ov_sum <- left_join(as.data.frame(bins_gr), ov_sum, by = c("idx" = "Var1"))

In [ ]:
# Fill bins with no overlapping SNV: NA -> 0.
ov_sum[is.na(ov_sum$Freq),]$Freq <- 0

In [ ]:
# Plotting library used by the next cell.
library(ggplot2)

In [ ]:
# Quick distribution sanity check: stacked-fill bar showing the fraction of
# 10 kb bins with each integer SNV count (0, 1, 2, ...). Most genome-wide
# bins carry 0 SNVs; the long tail at high counts represents SNV hotspots.
table(ov_sum$Freq) %>%
  as.data.frame() %>%
  ggplot(., aes(x = "identity", y = Freq)) +
    geom_col(aes(fill = Var1), color = "black", position = "fill")

In [ ]:
head(ov_sum)

In [ ]:
# OUTPUT 1: per-10kb-bin SNV count, full table (with chrom/start/end/idx/Freq).
write.table(ov_sum, "./GM12878_10k_snp_counts_summary.txt",
            row.names = F, quote = F, sep = '\t')

In [ ]:
# OUTPUT 2: same data as a 4-column bedGraph (chrom, start, end, Freq) so it
# can be loaded directly into IGV/UCSC as a track. Columns 1:3 = coords,
# column 7 of ov_sum = Freq (after as.data.frame(GRanges) the layout is
# seqnames, start, end, width, strand, idx, Freq).
write.table(ov_sum[,c(1:3,7)], "./GM12878_10k_snp_counts_summary.bedGraph",
            row.names = F, quote = F, col.names = F, sep = '\t')

In [ ]:
# Sanity: full 10 kb bin GRanges (~309k bins covering all of GRCh38).
bins_gr

In [ ]:
# Subset 10 kb bins to those that contain at least one SNV (~224k bins).
# This is the *coverage set*: each entry is a 10 kb window that we know
# carries phasing information. Used downstream to compute, per 500 kb
# super-bin, how many of its 50 child 10 kb bins are informative.
bin_snp <- bins_gr[unique(ov$query.id)]

In [ ]:
bin_snp

In [ ]:
# Build 500 kb tiles and retain chr1-22 plus chrX.
windows_500k <- tileGenome(
  seqinfo_hg38,
  tilewidth = 500000,
  cut.last.tile.in.chrom = TRUE
)
bins_500k <- as.data.frame(windows_500k)[, c("seqnames", "start", "end")]
colnames(bins_500k) <- c("chrom", "start", "end")
bins_500k$start <- bins_500k$start - 1L
bins_500k <- bins_500k[bins_500k$chrom %in% paste0("chr", c(1:22, "X")), ]
bins_500k$chrom <- factor(bins_500k$chrom, levels = paste0("chr", c(1:22, "X")))
bins_500k <- bins_500k %>% arrange(chrom, start)
bins_500k$idx <- seq_len(nrow(bins_500k))
bins_500k_gr <- dt2gr(bins_500k)


In [ ]:
# For each 500 kb bin, count how many of its child 10 kb bins (from bin_snp)
# carry at least one SNV. The metric we end up with is a *coverage fraction*
# at 10 kb resolution — range 0..50 — not a raw SNV count. This penalises
# 500 kb bins where SNVs are clustered at one spot but absent elsewhere,
# which would otherwise look "rich" if you only counted SNVs.
#
# `filter(width > 1)` drops zero-width adjacency hits at bin boundaries
# (gUtils' overlap operator emits a degenerate width=1 row when only the
# boundary touches; filter to honest interior overlaps).
# `n_distinct(subject.id)` == number of 10 kb bins (subject.id == bin_snp idx)
# that overlap this 500 kb super-bin (query.id == bins_500k_gr idx).
snp_sum <- bins_500k_gr %*% bin_snp %>% as.data.frame() %>%
  filter(width > 1) %>%
  dplyr::group_by(query.id) %>%
  dplyr::summarise(sum = n_distinct(subject.id))

In [ ]:
# Re-attach bin coordinates to the per-bin SNV-coverage count.
snp_sum <- left_join(bins_500k, snp_sum, by = c("idx" = "query.id"))

In [ ]:
# 500 kb bins with no SNV-bearing children: NA -> 0.
snp_sum[is.na(snp_sum$sum),]$sum <- 0

In [ ]:
head(snp_sum)

In [ ]:
# Genome-wide track of SNV-bearing 10 kb bins per 500 kb super-bin.
# Alternating chromosome colours (black/grey, with chrX in dark red) make the
# karyotype layout obvious. Useful eyeball check before sliding-window step.
options(repr.plot.width = 16, repr.plot.height = 2)
library(ggplot2)
ggplot(snp_sum, aes(x = idx, y = sum)) +
  geom_point(aes(color = chrom)) +
  scale_color_manual(values = c(rep(c("black", "grey"), 11), "darkred")) +
  theme_bw() +
  theme(axis.text.x = element_blank(), axis.title.x = element_blank()) +
  ylab("Number of SNV-bearing 10kb bins per 500kb bin") +
  ggtitle("SNV coverage per 500kb bin (NA12878 phased het SNVs)")


In [ ]:
# Re-print head as a sanity check before the sliding-window step.
head(snp_sum)

In [ ]:
library(dplyr)
library(furrr)
library(future.apply)

# ---------------------------------------------------------------------------
# sliding_window_genome
# ---------------------------------------------------------------------------
# Aggregate a per-bin numeric column over a sliding window of consecutive
# bins, separately for each chromosome. With the defaults below
# (window_size = 4, step = 1) and an input of 500 kb tiles, this produces
# overlapping 2 Mb windows stepped every 500 kb. We pick 2 Mb because the
# downstream C.Origami model takes a 2 Mb context per prediction.
#
# Args:
#   df          : data.frame with required columns chrom / start / end and a
#                 numeric column to summarise.
#   window_size : number of consecutive child bins per output window.
#   step        : stride (in child-bin units).
#   val_col     : index of the numeric column in `df` (default 5 -> 'sum').
#
# Returns: data.frame(chrom, start, end, mean_value), one row per window.
sliding_window_genome <- function(df, window_size = 4, step = 1, val_col = 5) {

  # Required columns check.
  if (!all(c("chrom", "start", "end") %in% colnames(df))) {
    stop("input data.frame must contain chrom/start/end columns")
  }

  # Sort by (chrom, start) so windowing walks each chromosome left-to-right.
  df_sorted <- df %>%
    arrange(chrom, start) %>%
    group_by(chrom) %>%
    mutate(row_id = row_number()) %>%
    ungroup()

  # Per-chromosome window builder. Returns one row per valid window.
  process_chrom <- function(sub_df) {
    n <- nrow(sub_df)
    if (n < window_size) return(data.frame())  # skip chromosomes shorter than 1 window

    # Valid window start indices: 1, 1+step, 1+2*step, ... up to (n-window_size+1).
    window_starts <- seq(1, n - window_size + 1, by = step)

    # Parallelise over windows within a chromosome (future_lapply backend).
    future_lapply(window_starts, function(i) {
      window_data <- sub_df[i:(i + window_size - 1), ]
      data.frame(
        chrom      = unique(window_data$chrom),
        start      = min(window_data$start),
        end        = max(window_data$end),
        mean_value = mean(window_data[[val_col]], na.rm = TRUE),
        stringsAsFactors = FALSE
      )
    }) %>% bind_rows()
  }

  # Split by chromosome, then map per-chromosome processing in parallel
  # (future_map_dfr respects the active future::plan).
  result <- df_sorted %>%
    group_split(chrom) %>%
    future_map_dfr(process_chrom, .progress = TRUE)

  # If every value in a window was NA, mean(...) returns NaN — convert to NA
  # so downstream filters/sorts behave consistently.
  result$mean_value[is.nan(result$mean_value)] <- NA

  return(result)
}

### Smoke test ---------------------------------------------------------------
# Tiny synthetic input: two chromosomes, 10 bins each, normal-distributed
# values around two distinct means. Confirms the function returns rolling
# means without dropping rows.
set.seed(123)
test_data <- data.frame(
  chrom = rep(c("chr1", "chr2"), each = 10),
  start = c(1:10 * 100, 1:10 * 150),
  end   = c(1:10 * 100 + 99, 1:10 * 150 + 149),
  value = c(rnorm(10, mean = 50), rnorm(10, mean = 100))
)

system.time({
  output <- sliding_window_genome(test_data, window_size = 4, val_col = 4)
})

print(head(output, 5))

In [ ]:
# Apply the sliding window over the 500 kb SNV-coverage track.
# val_col = 5 picks the 'sum' column (chrom/start/end/idx/sum). Defaults
# window_size=4, step=1 -> 2 Mb windows stepped every 500 kb.
snp_sum_sw <- sliding_window_genome(snp_sum)

In [ ]:
# Sort descending by mean_value: the most SNV-dense 2 Mb windows come first.
# `head -n N` of the output file then gives the top-N windows.
snp_sum_sw <- snp_sum_sw %>% arrange(-mean_value)

In [ ]:
# Top 50 windows. mean_value is "average # of SNV-bearing 10kb bins per
# 500kb sub-bin", so the theoretical max is 50 (every 10kb bin in the
# window carries a SNV). The top windows reach exactly 50 — fully informative.
head(snp_sum_sw, 50)

In [ ]:
# Write the ranked 2 Mb SNP-density regions to the shared repository input.
write.table(snp_sum_sw, summary_path,
            row.names = F, quote = F, sep = "\t")
